# Representative Frame Method Comparison

This notebook compares representative-frame selection rules for the `w/ live caption + w/o gaze annotation` multi-segment setting.

Methods included:
- `midpoint`
- `min_velocity`
- `local_avg_center`
- `hybrid_mid_gaze`
- `plateau_center`
- `equal_bin`

Current defaults:
- Savitzky-Golay: `window=11`, `poly=2`
- local average radius: `2`
- center fraction: `0.6`
- hybrid lambda: `0.75`
- plateau quantile: `0.25`
- equal-bin count: `5`

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

CSV_PATH = Path("debug/repframe_methods/repframe_methods_cap_true_gaze_false.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Missing CSV: {CSV_PATH}")

df = pd.read_csv(CSV_PATH).sort_values(["fps", "method"]).reset_index(drop=True)
display(df)


,method,num_evaluated_episodes,num_evaluated_intervals,exact_hit_rate,any_hit_episode_rate,all_interval_episode_rate,ordered_gt_hit_recall,ordered_all_hit_episode_rate,mean_nearest_gt_dist,hit@0_rate,hit@1_rate,hit@3_rate,hit@5_rate,result_file,fps,caption,gaze_annot,model
0,equal_bin,71,137,0.4599,0.6338,0.2676,0.4397,0.2535,3.09,0.4599,0.5182,0.7007,0.8175,result_multiseg_gemini-3.1-flash-lite-preview_...,2,True,False,gemini-3.1-flash-lite-preview
1,hybrid_mid_gaze,71,137,0.4964,0.6761,0.2958,0.4752,0.2817,2.96,0.4964,0.5620,0.7080,0.8175,result_multiseg_gemini-3.1-flash-lite-preview_...,2,True,False,gemini-3.1-flash-lite-preview
2,local_avg_center,71,137,0.5620,0.7324,0.3662,0.5390,0.3521,2.55,0.5620,0.5985,0.7299,0.8540,result_multiseg_gemini-3.1-flash-lite-preview_...,2,True,False,gemini-3.1-flash-lite-preview
3,midpoint,71,137,0.5401,0.7465,0.2958,0.5248,0.2958,2.23,0.5401,0.6204,0.8029,0.8467,result_multiseg_gemini-3.1-flash-lite-preview_...,2,True,False,gemini-3.1-flash-lite-preview
4,min_velocity,71,137,0.4891,0.7183,0.2535,0.4681,0.2254,3.22,0.4891,0.5401,0.7518,0.8175,result_multiseg_gemini-3.1-flash-lite-preview_...,2,True,False,gemini-3.1-flash-lite-preview
5,plateau_center,71,137,0.4672,0.6761,0.2535,0.4397,0.2254,2.88,0.4672,0.5620,0.7226,0.8394,result_multiseg_gemini-3.1-flash-lite-preview_...,2,True,False,gemini-3.1-flash-lite-preview
6,equal_bin,67,131,0.4122,0.5970,0.2090,0.4060,0.2090,6.38,0.4122,0.5191,0.7176,0.8015,result_multiseg_gemini-3.1-flash-lite-preview_...,4,True,False,gemini-3.1-flash-lite-preview
7,hybrid_mid_gaze,67,131,0.4733,0.6866,0.2537,0.4586,0.2388,6.24,0.4733,0.5191,0.6947,0.8015,result_multiseg_gemini-3.1-flash-lite-preview_...,4,True,False,gemini-3.1-flash-lite-preview
8,local_avg_center,67,131,0.5267,0.7612,0.2836,0.5113,0.2687,5.51,0.5267,0.6031,0.7786,0.8779,result_multiseg_gemini-3.1-flash-lite-preview_...,4,True,False,gemini-3.1-flash-lite-preview
9,midpoint,67,131,0.6183,0.8507,0.3881,0.6015,0.3582,5.26,0.6183,0.6870,0.8092,0.8550,result_multiseg_gemini-3.1-flash-lite-preview_...,4,True,False,gemini-3.1-flash-lite-preview


In [2]:
METHOD_ORDER = [
    "midpoint",
    "min_velocity",
    "local_avg_center",
    "hybrid_mid_gaze",
    "plateau_center",
    "equal_bin",
]

METHOD_LABELS = {
    "midpoint": "Midpoint",
    "min_velocity": "Min Velocity",
    "local_avg_center": "Local Avg + Center",
    "hybrid_mid_gaze": "Hybrid Mid+Gaze",
    "plateau_center": "Plateau Center",
    "equal_bin": "Equal Bin",
}

def method_metric_table(metric: str) -> pd.DataFrame:
    table = df.pivot(index="method", columns="fps", values=metric)
    table = table.reindex(METHOD_ORDER)
    table.index = [METHOD_LABELS[name] for name in table.index]
    return table

def display_percent_table(metric: str, title: str):
    table = method_metric_table(metric)
    display(Markdown(f"## {title}"))
    display(table.style.format(lambda value: f"{value * 100:.1f}%").highlight_max(axis=0, color="#dff6dd"))


In [3]:
display_percent_table("exact_hit_rate", "Interval Exact Hit Rate")
display_percent_table("hit@1_rate", "Interval Hit@1 Rate")
display_percent_table("all_interval_episode_rate", "Episode All-Interval Hit Rate")
display_percent_table("ordered_gt_hit_recall", "Ordered GT Hit Recall")


## Interval Exact Hit Rate

fps,2,4
Midpoint,54.0%,61.8%
Min Velocity,48.9%,44.3%
Local Avg + Center,56.2%,52.7%
Hybrid Mid+Gaze,49.6%,47.3%
Plateau Center,46.7%,43.5%
Equal Bin,46.0%,41.2%


## Interval Hit@1 Rate

fps,2,4
Midpoint,62.0%,68.7%
Min Velocity,54.0%,51.9%
Local Avg + Center,59.9%,60.3%
Hybrid Mid+Gaze,56.2%,51.9%
Plateau Center,56.2%,54.2%
Equal Bin,51.8%,51.9%


## Episode All-Interval Hit Rate

fps,2,4
Midpoint,29.6%,38.8%
Min Velocity,25.4%,22.4%
Local Avg + Center,36.6%,28.4%
Hybrid Mid+Gaze,29.6%,25.4%
Plateau Center,25.4%,19.4%
Equal Bin,26.8%,20.9%


## Ordered GT Hit Recall

fps,2,4
Midpoint,52.5%,60.2%
Min Velocity,46.8%,43.6%
Local Avg + Center,53.9%,51.1%
Hybrid Mid+Gaze,47.5%,45.9%
Plateau Center,44.0%,42.1%
Equal Bin,44.0%,40.6%


In [4]:
summary_cols = [
    "method",
    "fps",
    "exact_hit_rate",
    "hit@1_rate",
    "all_interval_episode_rate",
    "ordered_gt_hit_recall",
    "mean_nearest_gt_dist",
]

summary_df = df[summary_cols].copy()
summary_df["method"] = summary_df["method"].map(METHOD_LABELS)
display(Markdown("## Compact Summary"))
display(summary_df.style.format({
    "exact_hit_rate": lambda value: f"{value * 100:.1f}%",
    "hit@1_rate": lambda value: f"{value * 100:.1f}%",
    "all_interval_episode_rate": lambda value: f"{value * 100:.1f}%",
    "ordered_gt_hit_recall": lambda value: f"{value * 100:.1f}%",
    "mean_nearest_gt_dist": lambda value: f"{value:.2f}",
}))


## Compact Summary

,method,fps,exact_hit_rate,hit@1_rate,all_interval_episode_rate,ordered_gt_hit_recall,mean_nearest_gt_dist
0,Equal Bin,2,46.0%,51.8%,26.8%,44.0%,3.09
1,Hybrid Mid+Gaze,2,49.6%,56.2%,29.6%,47.5%,2.96
2,Local Avg + Center,2,56.2%,59.9%,36.6%,53.9%,2.55
3,Midpoint,2,54.0%,62.0%,29.6%,52.5%,2.23
4,Min Velocity,2,48.9%,54.0%,25.4%,46.8%,3.22
5,Plateau Center,2,46.7%,56.2%,25.4%,44.0%,2.88
6,Equal Bin,4,41.2%,51.9%,20.9%,40.6%,6.38
7,Hybrid Mid+Gaze,4,47.3%,51.9%,25.4%,45.9%,6.24
8,Local Avg + Center,4,52.7%,60.3%,28.4%,51.1%,5.51
9,Midpoint,4,61.8%,68.7%,38.8%,60.2%,5.26
